# Evaluation: parsing (VLM only vs Docling anchor)

## Setup

In [ ]:
import json, re, unicodedata
from pathlib import Path
import pandas as pd
import numpy as np

ROOT   = Path.cwd().parent
PARSED = ROOT / "data/parsed/machine_learning"
GOLDEN = ROOT / "data/test/golden"
ANCHOR = ROOT / "data/test/anchor"
EVAL_OUT = ROOT / "data/eval"
EVAL_OUT.mkdir(parents=True, exist_ok=True)

CONFIGS = [
    {"vorlesung": "SVM", "methode": "ohne Docling",
     "parse":  PARSED / "ML_5_svm/ML_5_svm_chunks.json",
     "golden": GOLDEN / "svm_golden_structured.json"},
    {"vorlesung": "SVM", "methode": "mit Docling",
     "parse":  ANCHOR / "ML_5_svm_chunks_docling_anker.json",
     "golden": GOLDEN / "svm_golden_structured.json"},
    {"vorlesung": "Neuronale Netze", "methode": "ohne Docling",
     "parse":  PARSED / "ML_9_neuronale_netze/ML_9_neuronale_netze_chunks_Reasoning_Off.json",
     "golden": GOLDEN / "ML_9_neuronale_netze_golden_structured.json"},
    {"vorlesung": "Neuronale Netze", "methode": "mit Docling",
     "parse":  ANCHOR / "ML_9_neuronale_netze_chunks_docling_anker.json",
     "golden": GOLDEN / "ML_9_neuronale_netze_golden_structured.json"},
]

def load_pairs(cfg):
    parse  = json.loads(Path(cfg["parse"]).read_text(encoding="utf-8"))
    golden = json.loads(Path(cfg["golden"]).read_text(encoding="utf-8"))
    by_id  = {c["id"]: c for c in parse}
    pairs  = [(g, by_id[g["slide_id"]]) for g in golden if g["slide_id"] in by_id]
    return parse, golden, by_id, pairs

print("Configurations:")
for cfg in CONFIGS:
    parse, golden, by_id, pairs = load_pairs(cfg)
    print(f"  {cfg['vorlesung']:16s} | {cfg['methode']:13s} | "
          f"parse={len(parse):3d}  golden={len(golden):3d}  pairs={len(pairs):3d}")

## Normalise

In [ ]:

LATEX = {
    r"\alpha": "α", r"\beta": "β", r"\gamma": "γ", r"\delta": "δ",
    r"\epsilon": "ε", r"\varepsilon": "ε", r"\zeta": "ζ", r"\eta": "η",
    r"\theta": "θ", r"\kappa": "κ", r"\lambda": "λ", r"\mu": "μ",
    r"\nu": "ν", r"\xi": "ξ", r"\pi": "π", r"\rho": "ρ", r"\sigma": "σ",
    r"\tau": "τ", r"\phi": "φ", r"\chi": "χ", r"\psi": "ψ", r"\omega": "ω",
    r"\leq": "≤", r"\le": "≤", r"\geq": "≥", r"\ge": "≥",
    r"\neq": "≠", r"\ne": "≠", r"\approx": "≈", r"\times": "×",
    r"\pm": "±", r"\infty": "∞", r"\sum": "∑", r"\partial": "∂",
    r"\nabla": "∇", r"\in": "∈", r"\rightarrow": "→", r"\to": "→",
}

def normalize(t: str) -> str:
    t = unicodedata.normalize("NFC", t)
    t = t.lower()
    for cmd in sorted(LATEX, key=len, reverse=True):    
        t = re.sub(re.escape(cmd) + r"(?![a-z])", LATEX[cmd], t)
    t = t.replace(",,", "").replace("``", "").replace("''", "")  
    t = re.sub(r'[„“”‚‘’»«"]', "", t)                    
    t = re.sub(r"\$+", " ", t)                           
    t = re.sub(r"[*#`>~]", " ", t)                   
    t = re.sub(r"…", "...", t)                          
    t = re.sub(r"[‐-―−]", "-", t)         
    t = re.sub(r"[•·‣▪]", " ", t)                       
    t = re.sub(r"(?m)^\s*[-→]\s+", " ", t)               
    t = re.sub(r";", " ", t)                            
    t = re.sub(r"\s*([.,:!?=≠≥≤≈×±])\s*", r"\1", t)     
    t = re.sub(r"\s+", " ", t)                        
    return t.strip()

## Extract text and blocks from chunks

Defining helpers to split a chunk into plain text nuggets versus [GRAFIK]/[FORMEL]/[CODE] blocks, isolating the text content for string based recall

In [ ]:
BLOCK_PREFIXES = ("[GRAFIK]", "[FORMEL]", "[CODE]")

def is_text_nugget(n: str) -> bool:
    return not n.lstrip().startswith(BLOCK_PREFIXES)

def text_only(page_content: str) -> str:
    content = page_content.replace("\\n", "\n")         
    blocks = re.split(r"\n\s*\n", content)
    return "\n\n".join(b for b in blocks if not b.lstrip().startswith(BLOCK_PREFIXES))

def build_parsetext(chunk: dict) -> str:
    parts = []
    if chunk.get("title"):
        parts.append("Titel: " + chunk["title"])
    parts.append(text_only(chunk.get("page_content", "")))
    return "\n".join(parts)

## Metric: recall

How many of a slide's gold text facts show up (as a normalised substring) in the parsed chunk

In [ ]:
def recall_counts(text_nuggets, chunk):
    if not text_nuggets:
        return 0, 0
    c = normalize(chunk)
    hits = sum(normalize(n) in c for n in text_nuggets)
    return hits, len(text_nuggets)

## Recall over all texts

Computing pooled text recall per config, comparing VLM only against Docling anchored parsing on plain text

In [ ]:
text_rows = []          
per_slide = {}         

for cfg in CONFIGS:
    _, _, _, pairs = load_pairs(cfg)
    rows = []
    tot_hits = 0
    tot_n = 0
    
    for g, p in pairs:
        text_nugget = [n for n in g["text"] if is_text_nugget(n)]
        chunk = build_parsetext(p)
        hits, n = recall_counts(text_nugget, chunk)
        tot_hits += hits
        tot_n    += n
        slide = g["slide_id"].split("_page_")[-1]
        recall = hits / n if n else None

        rows.append(
            {
                "slide": slide,
                "text": n,
                "hits": hits,
                "recall": recall,
            }
        )

    per_slide[(cfg["vorlesung"], cfg["methode"])] = pd.DataFrame(rows)
    recall = tot_hits / tot_n if tot_n else float("nan")
    text_rows.append({
        "Vorlesung": cfg["vorlesung"], "Methode": cfg["methode"],
        "Text-Recall": recall, "Text-Fakten": tot_n,
    })

text_df = pd.DataFrame(text_rows)
text_df

## LLM as a judge: setup

Configuring the LLM as judge client, used for the block modalities where exact string matching fails (formulas, code)

In [ ]:
import os
import re
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(os.path.join(os.getcwd(), "..", ".env"), override=True)

GATEWAY_URL = os.getenv("GATEWAY_URL", "")
BEARER_TOKEN = os.getenv("BEARER_TOKEN", "")
JUDGE_MODEL = os.getenv("INFERENCE_MODEL_GATEWAY", "") 

client = OpenAI(base_url=GATEWAY_URL, api_key=BEARER_TOKEN)

print("Judge-LLM Model:", JUDGE_MODEL)

An LLM call (with retries) that decides semantically whether a gold element is covered by the parsed slide (covered or missing)

In [ ]:

BLOCK_TYPES = ["grafik", "formel", "code"]

def build_fulltext(chunk):
    titel = "Titel: " + chunk["title"] + "\n" if chunk.get("title") else ""
    return titel + chunk.get("page_content", "")

cache = {}

def call_judge(prompt):
    kwargs = dict(
        model=JUDGE_MODEL,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    try:
        return client.chat.completions.create(
            response_format={"type": "json_object"}, **kwargs
        )
    except Exception:
        return client.chat.completions.create(**kwargs)

def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n?", "", text)
        text = re.sub(r"\n?```$", "", text).strip()
    m = re.search(r"\{.*\}", text, re.DOTALL)   
    if m:
        text = m.group(0)
    return json.loads(text)

def judge_item(element, parse_text, retries=2):
    key = (element, parse_text)
    if key in cache:
        return cache[key]

    prompt = f"""
        Du bist ein Evaluator fuer Informations-Recall in Folien-Parsing.

        Aufgabe:
        Pruefe ob, die Information des gegebenen Elements irgendwo im geparsten Text enthalten ist.

        Wichtig:
        - Es spielt KEINE Rolle, wo die Information steht (Text, Grafikbeschreibung, Formel, Code).
        - Es spielt KEINE Rolle, wie sie formuliert ist.
        - Entscheidend ist nur semantische Ähnlichkeit.
        - Zusaetzlicher Inhalt im Parse ist irrelevant.
        - Du bewertest nur Recall: enthalten oder nicht enthalten.

        Element:
        {element}

        Geparste Folie:
        {parse_text}

        Antworte strikt als JSON:
        {{"verdict": "covered" oder "missing", "reason": "kurze Begründung"}}
        """

    last_err = None
    for _ in range(retries):
        try:
            antwort = call_judge(prompt)
            parsed = extract_json(antwort.choices[0].message.content)
            verdict = str(parsed.get("verdict", "")).strip().lower()
            if verdict not in ("covered", "missing"):
                raise ValueError(f"unerwartetes verdict: {verdict!r}")
            ergebnis = {"verdict": verdict, "reason": parsed.get("reason", "")}
            cache[key] = ergebnis         
            return ergebnis
        except Exception as e:
            last_err = e

    return {"verdict": "invalid", "reason": f"Judge unbrauchbar nach {retries} Versuchen: {last_err}"}


## Block recall: formula and code

Running the judge to measure formula and code recall per config, the modalities that need semantic matching rather than string matching

In [ ]:
RUN_JUDGE = True
JUDGE_TYPES = ["formel", "code"]

block_rows = []   

if RUN_JUDGE:
    for cfg in CONFIGS:
        _, golden, by_id, _ = load_pairs(cfg)

        for modality in JUDGE_TYPES:
            covered = 0
            total = 0
            for g in golden:
                sid = g["slide_id"]
                if sid not in by_id:
                    continue
                parse_text = build_fulltext(by_id[sid])
                for element in g.get(modality, []):
                    verdict = judge_item(element, parse_text)["verdict"]
                    if verdict in ("covered", "missing"):
                        total += 1
                        if verdict == "covered":
                            covered += 1

            recall = covered / total if total else None
            block_rows.append({
                "Vorlesung": cfg["vorlesung"],
                "Methode": cfg["methode"],
                "Modalität": modality,
                "Recall": recall,
                "Treffer": covered,
                "Total": total,
            })
            recall_txt = f"{recall:.3f}" if recall is not None else "-"
            print(f"  {cfg['vorlesung']:16s} | {cfg['methode']:13s} | "
                  f"{modality:6s} {covered:3d}/{total:3d} = {recall_txt}")

block_df = pd.DataFrame(block_rows)
block_df

## Overall table: recall with and without Docling

Pooling text/formula/code recall into one table with and without Docling, including the deltas

In [ ]:
methoden = ["ohne Docling", "mit Docling"]

def text_pooled(meth):
    hits = total = 0
    for cfg in CONFIGS:
        if cfg["methode"] != meth:
            continue
        _, _, _, pairs = load_pairs(cfg)
        for g, p in pairs:
            tn = [n for n in g["text"] if is_text_nugget(n)]
            h, n = recall_counts(tn, build_parsetext(p))
            hits += h
            total += n
    return hits, total

def block_pooled(meth, modality):

    sub = block_df[(block_df["Methode"] == meth) & (block_df["Modalität"] == modality)]
    return int(sub["covered"].sum()), int(sub["total"].sum())

counts = {}
for meth in methoden:
    counts[(meth, "Text")]   = text_pooled(meth)
    counts[(meth, "Formel")] = block_pooled(meth, "formel")
    counts[(meth, "Code")]   = block_pooled(meth, "code")

modality = ["Text", "Formel", "Code"]
rows = []
for mod in modality:
    o_cov, o_tot = counts[("ohne Docling", mod)]
    m_cov, m_tot = counts[("mit Docling", mod)]
    o_rec, m_rec = o_cov / o_tot, m_cov / m_tot
    rows.append({"Modalität": mod, "n": o_tot,
                 "ohne Docling": o_rec, "mit Docling": m_rec,
                 "Δ (mit − ohne)": m_rec - o_rec})

o_cov = sum(counts[("Recall - ohne Docling", m)][0] for m in modality)
o_tot = sum(counts[("Recall - ohne Docling", m)][1] for m in modality)
m_cov = sum(counts[("Recall - mit Docling", m)][0] for m in modality)
m_tot = sum(counts[("Recall - mit Docling", m)][1] for m in modality)
rows.append({"Modalität": "Gesamt", "n": o_tot,
             "ohne Docling": o_cov / o_tot, "mit Docling": m_cov / m_tot,
             "Δ (mit − ohne)": m_cov / m_tot - o_cov / o_tot})

cmp_df = pd.DataFrame(rows)
cmp_df.to_csv(EVAL_OUT / "parsing_docling_vergleich.csv", index=False, encoding="utf-8")
print("gespeichert:", EVAL_OUT / "parsing_docling_vergleich.csv")

cmp_df.style.format({"n": "{:.0f}", "Recall - ohne Docling": "{:.3f}",
                     "Recall - mit Docling": "{:.3f}", "Δ (mit − ohne)": "{:+.3f}"}).hide(axis="index")